# API Predictions Validation Notebook

This notebook generates predictions using the same aggregation strategy as the API, producing a single label column for each product.

## Purpose
- Generate predictions for all products using the API's aggregation logic
- Handle duplicate products with mean/mode aggregation
- Create CSV with aggregated labels (success/failure)

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import joblib
import warnings
from collections import Counter
import sys
import os

# Add app directory to path
sys.path.append('app')
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


## Step 1: Load Data and Models

In [2]:
# Load the data
print("Loading datasets...")
engineered_df = pd.read_csv('data/processed/ecommerce_sales_engineered.csv')
featured_df = pd.read_csv('data/processed/ecommerce_sales_featured.csv')

# Ensure product_id is the same type in both dataframes
engineered_df['product_id'] = engineered_df['product_id'].astype(int)
featured_df['product_id'] = featured_df['product_id'].astype(int)

# Remove any existing success column from engineered_df to avoid conflicts
if 'success' in engineered_df.columns:
    engineered_df = engineered_df.drop('success', axis=1)

# Create a clean subset of featured_df with just the columns we need
featured_subset = featured_df[['product_id', 'success']].copy()
featured_subset['success'] = featured_subset['success'].astype(int)

print("\nBefore merge:")
print(f"Featured subset columns: {featured_subset.columns.tolist()}")
print(f"Engineered df columns: {len(engineered_df.columns)} columns")

# Perform the merge
df_with_labels = pd.merge(
    engineered_df,
    featured_subset,
    on='product_id',
    how='left'
)

print("\nAfter merge:")
print(f"Total products: {len(df_with_labels)}")
print(f"Success column exists: {'success' in df_with_labels.columns}")

# Verify success column data
success_count = df_with_labels['success'].sum()
fail_count = (df_with_labels['success'] == 0).sum()
print(f"\nSuccess products: {success_count}")
print(f"Fail products: {fail_count}")
print(f"Columns in dataset: {df_with_labels.shape[1]}")

Loading datasets...

Before merge:
Featured subset columns: ['product_id', 'success']
Engineered df columns: 116 columns

After merge:
Total products: 1000
Success column exists: True

Success products: 500
Fail products: 500
Columns in dataset: 117


In [3]:
# Load the model components
model_package = joblib.load('models/final/all_models_phase6.pkl')

all_models = model_package['models']
scaler = model_package['scaler']
feature_columns = model_package['feature_columns']
best_model_name = model_package['best_model']

# Extract the calibrated ensemble model
calibrated_ensemble_data = all_models[best_model_name]
if isinstance(calibrated_ensemble_data, dict) and 'model' in calibrated_ensemble_data:
    primary_model = calibrated_ensemble_data['model']
else:
    primary_model = calibrated_ensemble_data

print(f"Model loaded: {best_model_name}")
print(f"Feature columns: {len(feature_columns)}")

Model loaded: calibrated_ensemble
Feature columns: 114


## Step 2: Define Prediction Functions (Same as API)

In [4]:
def smooth_probability(p, temperature=2.0):
    """Apply temperature scaling and bounds to smooth extreme probabilities"""
    # Apply temperature scaling to reduce confidence
    logit = np.log(p / (1 - p + 1e-10))
    scaled_logit = logit / temperature
    smoothed = 1 / (1 + np.exp(-scaled_logit))
    
    # Apply additional bounds to prevent extreme predictions
    if smoothed < 0.02:
        smoothed = 0.1 + (smoothed * 4)  # Map 0-2% to 10-18%
    elif smoothed < 0.1:
        smoothed = 0.15 + (smoothed - 0.02) * 2  # Map 2-10% to 15-31%
    elif smoothed > 0.98:
        smoothed = 0.82 + (1 - smoothed) * 4  # Map 98-100% to 82-90%
    elif smoothed > 0.9:
        smoothed = 0.75 + (smoothed - 0.9) * 0.7  # Map 90-98% to 75-82%
    
    return smoothed

def get_prediction_for_row(row, feature_columns, scaler, model):
    """Get prediction for a single product row"""
    # Extract features in correct order
    feature_values = []
    for col in feature_columns:
        if col in row.index:
            value = row[col]
            if pd.isna(value):
                value = 0
            feature_values.append(value)
        else:
            feature_values.append(0)
    
    # Create feature array and scale
    features_array = np.array(feature_values).reshape(1, -1)
    if scaler is not None:
        features_array = scaler.transform(features_array)
    
    # Get prediction
    pred_proba = model.predict_proba(features_array)
    if len(pred_proba.shape) > 1 and pred_proba.shape[1] > 1:
        probability = pred_proba[0, 1]
    else:
        probability = pred_proba[0]
    
    # Apply smoothing
    probability = smooth_probability(probability)
    
    return probability

print("Prediction functions defined")

Prediction functions defined


## Step 3: Generate Predictions for All Products

In [5]:
# Get all unique product names
unique_products = df_with_labels['product_name'].unique()
print(f"Total unique product names: {len(unique_products)}")

# Count products with duplicates
product_counts = df_with_labels['product_name'].value_counts()
products_with_duplicates = product_counts[product_counts > 1]
print(f"Product names with duplicates: {len(products_with_duplicates)}")
print(f"\nTop 5 products with most duplicates:")
print(products_with_duplicates.head())

Total unique product names: 61
Product names with duplicates: 61

Top 5 products with most duplicates:
product_name
Generic Product    139
Monitor             23
Dumbbells           22
Action Figure       21
Self-Help Book      21
Name: count, dtype: int64


In [6]:
# Generate predictions for all products
results = []

for product_name in unique_products:
    # Get all rows for this product
    product_rows = df_with_labels[df_with_labels['product_name'] == product_name]
    
    if len(product_rows) == 1:
        # SINGLE PRODUCT - Direct prediction
        row = product_rows.iloc[0]
        probability = get_prediction_for_row(row, feature_columns, scaler, primary_model)
        label = 'Success' if probability >= 0.5 else 'Fail'
        
        results.append({
            'product_name': product_name,
            'product_id': row['product_id'],
            'num_variants': 1,
            'aggregation_method': 'Single',
            'probability': probability,
            'label': label,
            'category': row['category'],
            'price': row['price'],
            'total_sales': row[[f'sales_month_{i}' for i in range(1, 13)]].sum(),
            'review_score': row['review_score'],
            'review_count': row['review_count']
        })
        
    else:
        # MULTIPLE PRODUCTS - Use aggregation
        individual_probs = []
        individual_preds = []
        
        for _, row in product_rows.iterrows():
            prob = get_prediction_for_row(row, feature_columns, scaler, primary_model)
            pred = 1 if prob >= 0.5 else 0
            
            individual_probs.append(prob)
            individual_preds.append(pred)
        
        # Apply aggregation rules
        # Rule 1: Try MODE
        pred_counts = Counter(individual_preds)
        most_common = pred_counts.most_common(1)[0]
        
        # Check if there's a clear winner
        if len(pred_counts) == 1 or (len(pred_counts) == 2 and 
                                     pred_counts.most_common(2)[0][1] != 
                                     pred_counts.most_common(2)[1][1]):
            # Use MODE
            final_prediction = most_common[0]
            # Average probability of the winning class
            final_probability = np.mean([p for i, p in enumerate(individual_probs) 
                                        if individual_preds[i] == final_prediction])
            aggregation_method = 'Mode'
        else:
            # TIE - Use MEAN
            final_probability = np.mean(individual_probs)
            final_prediction = 1 if final_probability >= 0.5 else 0
            aggregation_method = 'Mean (tie)'
        
        # Set label based on aggregated prediction
        label = 'Success' if final_prediction == 1 else 'Fail'
        
        # Get average values for other features
        avg_price = product_rows['price'].mean()
        avg_sales = product_rows[[f'sales_month_{i}' for i in range(1, 13)]].sum(axis=1).mean()
        avg_review_score = product_rows['review_score'].mean()
        avg_review_count = product_rows['review_count'].mean()
        category = product_rows['category'].mode()[0]
        
        results.append({
            'product_name': product_name,
            'product_id': f"Multiple ({len(product_rows)} variants)",
            'num_variants': len(product_rows),
            'aggregation_method': aggregation_method,
            'probability': final_probability,
            'label': label,
            'category': category,
            'price': avg_price,
            'total_sales': avg_sales,
            'review_score': avg_review_score,
            'review_count': avg_review_count,
            'predicted_success_rate': np.mean(individual_preds)
        })

print(f"Processed {len(results)} unique product names")

Processed 61 unique product names


## Step 4: Create Results DataFrame

In [7]:
# Create DataFrame
results_df = pd.DataFrame(results)

# Convert probability to percentage
results_df['probability_percentage'] = (results_df['probability'] * 100).round(2)

# Reorder columns for better readability
column_order = [
    'product_name',
    'product_id',
    'num_variants',
    'aggregation_method',
    'probability_percentage',
    'label',  # Single label column showing the aggregated result
    'category',
    'price',
    'total_sales',
    'review_score',
    'review_count'
]

# Add extra columns for products with duplicates
if 'predicted_success_rate' in results_df.columns:
    column_order.append('predicted_success_rate')

# Select and reorder columns
final_df = results_df[column_order]

print("Results DataFrame created")
print(f"Shape: {final_df.shape}")
print(f"\nFirst few rows:")
final_df.head(10)

Results DataFrame created
Shape: (61, 12)

First few rows:


,product_name,product_id,num_variants,aggregation_method,probability_percentage,label,category,price,total_sales,review_score,review_count,predicted_success_rate
0,Hoodie,Multiple (12 variants),12,Mean (tie),48.29,Fail,Clothing,193.555833,5826.166667,2.608333,412.333333,0.500000
1,Cookware Set,Multiple (13 variants),13,Mode,75.86,Success,Home & Kitchen,309.765385,6112.307692,3.130769,617.307692,0.538462
2,Train Set,Multiple (9 variants),9,Mode,79.15,Success,Toys,310.242222,6422.888889,2.455556,342.222222,0.666667
3,Remote Car,Multiple (15 variants),15,Mode,23.17,Fail,Toys,260.468667,6063.266667,3.240000,516.600000,0.466667
4,Self-Help Book,Multiple (21 variants),21,Mode,78.74,Success,Books,241.805714,6293.428571,3.114286,469.904762,0.571429
5,Action Figure,Multiple (21 variants),21,Mode,75.41,Success,Toys,253.156667,5957.428571,3.038095,609.142857,0.523810
6,Laptop,Multiple (11 variants),11,Mode,21.42,Fail,Electronics,245.109091,5840.363636,3.200000,432.818182,0.363636
7,Camera,Multiple (20 variants),20,Mode,79.72,Success,Electronics,212.202500,6239.550000,3.025000,554.900000,0.550000
8,Shirt,Multiple (14 variants),14,Mean (tie),54.77,Success,Clothing,296.170714,5987.642857,2.928571,591.428571,0.500000
9,Building Blocks,Multiple (14 variants),14,Mode,28.91,Fail,Toys,248.221429,5743.142857,2.457143,423.785714,0.357143


## Step 5: Analyze Label Distribution

In [8]:
# Label distribution
print("Label Distribution:")
label_counts = results_df['label'].value_counts()
print(label_counts)
print(f"\nSuccess Rate: {(label_counts.get('Success', 0) / len(results_df)) * 100:.2f}%")
print(f"Fail Rate: {(label_counts.get('Fail', 0) / len(results_df)) * 100:.2f}%")
print()

# Distribution by aggregation method
print("Label Distribution by Aggregation Method:")
label_by_method = results_df.groupby(['aggregation_method', 'label']).size().unstack(fill_value=0)
print(label_by_method)
print()

# Average probability by label
print("Average Probability by Label:")
avg_prob_by_label = results_df.groupby('label')['probability_percentage'].agg(['mean', 'std', 'min', 'max'])
avg_prob_by_label = avg_prob_by_label.round(2)
print(avg_prob_by_label)

Label Distribution:
label
Success    36
Fail       25
Name: count, dtype: int64

Success Rate: 59.02%
Fail Rate: 40.98%

Label Distribution by Aggregation Method:
label               Fail  Success
aggregation_method               
Mean (tie)             2        5
Mode                  23       31

Average Probability by Label:
          mean   std    min    max
label                             
Fail     28.57  6.95  21.42  48.29
Success  73.08  9.00  50.39  80.15


## Step 6: Analyze Products with Duplicates

In [9]:
# Filter products with duplicates
duplicates_df = final_df[final_df['num_variants'] > 1].copy()

print(f"Products with duplicates: {len(duplicates_df)}")
print(f"\nAggregation method distribution:")
print(duplicates_df['aggregation_method'].value_counts())
print()

# Show examples of products with duplicates
print("Examples of products with multiple variants:")
print(duplicates_df.nlargest(5, 'num_variants')[[
    'product_name', 'num_variants', 'aggregation_method', 
    'probability_percentage', 'label'
]])
print()

# Products where aggregation method was Mean (tie)
tie_products = duplicates_df[duplicates_df['aggregation_method'] == 'Mean (tie)']
if len(tie_products) > 0:
    print(f"\nProducts where tie-breaking (mean) was used: {len(tie_products)}")
    print(tie_products[['product_name', 'num_variants', 'probability_percentage', 'label']])

Products with duplicates: 61

Aggregation method distribution:
aggregation_method
Mode          54
Mean (tie)     7
Name: count, dtype: int64

Examples of products with multiple variants:
       product_name  num_variants aggregation_method  probability_percentage  \
10  Generic Product           139               Mode                   75.97   
39          Monitor            23               Mode                   25.32   
34        Dumbbells            22               Mode                   76.72   
4    Self-Help Book            21               Mode                   78.74   
5     Action Figure            21               Mode                   75.41   

      label  
10  Success  
39     Fail  
34  Success  
4   Success  
5   Success  


Products where tie-breaking (mean) was used: 7
     product_name  num_variants  probability_percentage    label
0          Hoodie            12                   48.29     Fail
8           Shirt            14                   54.77  Success
18 

## Step 7: Export to CSV

In [11]:
# Save to CSV
output_filename = 'api_predictions_validation.csv'
final_df.to_csv(output_filename, index=False)

print(f"✅ Results saved to: {output_filename}")
print(f"\nFile contains:")
print(f"- {len(final_df)} unique products")
print(f"- {len(final_df[final_df['num_variants'] == 1])} single products")
print(f"- {len(final_df[final_df['num_variants'] > 1])} products with multiple variants")
print(f"\nColumns in CSV:")
for col in final_df.columns:
    print(f"  - {col}")
print(f"\nLabel Summary:")
print(f"- Success products: {len(final_df[final_df['label'] == 'Success'])}")
print(f"- Fail products: {len(final_df[final_df['label'] == 'Fail'])}")

# Show sample of the CSV
print("\nSample of exported data:")
print(final_df[['product_name', 'aggregation_method', 'probability_percentage', 'label']].head(10))

✅ Results saved to: api_predictions_validation.csv

File contains:
- 61 unique products
- 0 single products
- 61 products with multiple variants

Columns in CSV:
  - product_name
  - product_id
  - num_variants
  - aggregation_method
  - probability_percentage
  - label
  - category
  - price
  - total_sales
  - review_score
  - review_count
  - predicted_success_rate

Label Summary:
- Success products: 36
- Fail products: 25

Sample of exported data:
      product_name aggregation_method  probability_percentage    label
0           Hoodie         Mean (tie)                   48.29     Fail
1     Cookware Set               Mode                   75.86  Success
2        Train Set               Mode                   79.15  Success
3       Remote Car               Mode                   23.17     Fail
4   Self-Help Book               Mode                   78.74  Success
5    Action Figure               Mode                   75.41  Success
6           Laptop               Mode          